In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/raw_data.csv")

print(df.isnull().sum())
print("#"*50)
print(df.describe())

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64
##################################################
       SeniorCitizen       tenure  MonthlyCharges
count    7043.000000  7043.000000     7043.000000
mean        0.162147    32.371149       64.761692
std         0.368612    24.559481       30.090047
min         0.000000     0.000000       18.250000
25%         0.000000     9.000000       35.500000
50%         0.000000    29.000000       70.350000
75%         0.000000    55.000000       89.850000
max         1.000000    72.000000      118.750000


In [11]:

df = df.drop(columns=['customerID'])
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)


X = df.drop(columns=['Churn'])
y = df['Churn'].map({"Yes" : 1, "No" : 0})


X = pd.get_dummies(X, drop_first=True, dtype=int)
print(X.shape)  


xtrain, xtest, ytrain, ytest = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(len(xtrain.columns))   

(7043, 30)
30


In [12]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
xtrainr , ytrainr = smote.fit_resample(xtrain , ytrain)
print(ytrainr.value_counts())

Churn
0    4139
1    4139
Name: count, dtype: int64


In [16]:
test_df = pd.DataFrame(xtest)
test_df["Churn"] = pd.DataFrame(ytest)

train_df = pd.DataFrame(xtrainr)
train_df["Churn"] = pd.DataFrame(ytrainr)

test_df.to_csv("../data/test.csv")
train_df.to_csv("../data/train.csv")



Accuracy: 0.7643718949609652
              precision    recall  f1-score   support

          No       0.88      0.79      0.83      1035
         Yes       0.54      0.70      0.61       374

    accuracy                           0.76      1409
   macro avg       0.71      0.74      0.72      1409
weighted avg       0.79      0.76      0.77      1409

Feature importances: [0.00718132 0.11071156 0.07270372 0.09191941 0.0128534  0.02502465
 0.04799707 0.00432973 0.00456166 0.00650765 0.04215222 0.01608468
 0.01932842 0.08669191 0.01397796 0.03520566 0.01692463 0.01788913
 0.02777718 0.06890863 0.01360165 0.00634067 0.01561108 0.00602627
 0.06229966 0.10598084 0.00813535 0.02825006 0.00746584 0.01755801]


In [7]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale"))
model.fit(xtrainr, ytrainr)
y_pred = model.predict(xtest)

print("Accuracy:", accuracy_score(ytest, y_pred))
print(classification_report(ytest, y_pred))

Accuracy: 0.7629524485450674
              precision    recall  f1-score   support

          No       0.86      0.81      0.83      1035
         Yes       0.55      0.62      0.58       374

    accuracy                           0.76      1409
   macro avg       0.70      0.72      0.71      1409
weighted avg       0.77      0.76      0.77      1409



Model saved successfully!
